[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/martinloretzzz/vector-index-embedding/blob/main/quickstart.ipynb)

# Accelerating LLM Inference via Vector Index Based Output Embeddings

This notebook demonstrates how to accelerate LLM inference for compact models using vector index-based output embeddings, as described in our paper.

[Arxiv Link (online soon)]()

## Abstract

Large output embedding matrices create a significant memory bandwidth bottleneck during autoregressive decoding, especially for compact LLMs
with large multilingual vocabularies. We reformulate the output projection followed by top-k token
selection as a maximum inner product search over
token embeddings and replace the dense vocabulary projection with an HNSW-based vector index.
The resulting output head retrieves only a small
candidate set of high-scoring tokens and can be
integrated into existing decoding pipelines by scattering retrieved logits into a sparse full-vocabulary
tensor. On CPU inference with Gemma 3, Llama
3.2, and Qwen 3 models, our method substantially
accelerates the output projection and improves
end-to-end batch-size-one decoding throughput
by up to 82% for Gemma 3 270M, while preserving generation quality under AlpacaEval evaluation. These results suggest approximate retrieval
is a practical alternative to dense output projections in latency-sensitive small-batch decoding.

In [21]:
!pip install -q transformers vector-index-embedding

## Usage

Using the vector index embedding is straightforward and integrates seamlessly with Hugging Face transformers by simply replacing the model's lm_head. To try it out immediately, we provide prebuilt vector indices for several popular models on [Hugging Face](https://huggingface.co/martinloretzzz/vector-index-embedding).

In [22]:
from transformers import pipeline
from vectorindex import VectorIndexEmbedding

model_id = "Qwen/Qwen3-0.6B" # "google/gemma-3-270m-it"
pipe = pipeline("text-generation", model=model_id, device="cpu")
pipe.model.lm_head = VectorIndexEmbedding.from_pretrained(model_id, ef=200)

prompt = [{"role": "user", "content": "Who was Alan Turing?"}]
out = pipe(prompt)
print(out[0]["generated_text"][-1]["content"])

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<think>
Okay, the user is asking about Alan Turing. Let me start by confirming his full name and the context. Alan Turing was a British computer scientist and mathematician, right? He's well-known for the Turing test and the concept of the Halting Problem.

I should mention his contributions to computer science, like breaking the code during World War II, and his work on computers. Maybe include the Turing Test, which is a key concept in AI. Also, his work on computational theory and the development of the Turing machine are important points.

Wait, should I add his later work, like the development of the ENIAC? That's a good detail. Also, his legacy in both science and technology. Make sure to keep the explanation clear and concise, avoiding any jargon. Check if there's any specific information that's crucial for the answer. Yeah, that should cover his background, major contributions, and significance.
</think>

Alan Turing was a British computer scientist, mathematician, and logician

## Performance & Accuracy

We compare the performance and generated text of our vector index model to the unmodified baseline. You can increase the ef parameter for better recall, but higher search times. To get even better performance, you can use our modified version of hnswlib [martinloretzzz/hnswlib](https://github.com/martinloretzzz/hnswlib), but that might not work on all systems, as that code was only built and tested on our benchmarking server.

In [23]:
from transformers import pipeline
from vectorindex import VectorIndexEmbedding
import time
import torch

model_id = "google/gemma-3-270m-it" # "Qwen/Qwen3-0.6B"
new_token_count = 256
prompt = [{"role": "user", "content": "How does a computer work?"}]

pipe_ref = pipeline("text-generation", model=model_id, device="cpu", dtype=torch.float32)
pipe_ref.model.generation_config.cache_implementation = "static"
pipe_ref.model.forward = torch.compile(pipe_ref.model.forward, mode="reduce-overhead", backend="inductor")

pipe_vec = pipeline("text-generation", model=model_id, device="cpu", dtype=torch.float32)
pipe_vec.model.lm_head = VectorIndexEmbedding.from_pretrained(model_id, ef=200)
pipe_vec.model.generation_config.cache_implementation = "static"
pipe_vec.model.forward = torch.compile(pipe_vec.model.forward, mode="reduce-overhead", backend="inductor")


def timeit_wrapper(func, num_warmup=2, num_run=4):
    out = None
    for i in range(num_warmup):
        out_warmup = func()
    start = time.perf_counter()
    for i in range(num_run):
        out = func()
    end = time.perf_counter()
    t = (end - start) / num_run
    return out, t

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

In [24]:
# Baseline model
out, time_ref = timeit_wrapper(lambda: pipe_ref(prompt, max_new_tokens=new_token_count, min_new_tokens=new_token_count, do_sample=False, eos_token_id=None, pad_token_id=None))
text_ref = out[0]["generated_text"][1]["content"]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=256) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=256) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/do

In [25]:
# Vector Index Embedding Model
out, time_vec = timeit_wrapper(lambda: pipe_vec(prompt, max_new_tokens=new_token_count, min_new_tokens=new_token_count, do_sample=False, eos_token_id=None, pad_token_id=None))
text_vec = out[0]["generated_text"][1]["content"]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=256) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=256) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/do

In [26]:
print(f"Baseline Model:               time = {time_ref:.2f}s throughput = {new_token_count / time_ref:.2f} tok/s")
print(f"Vector Index Embedding Model: time = {time_vec:.2f}s throughput = {new_token_count / time_vec:.2f} tok/s")
print(f"Speedup: {time_ref / time_vec:.4f}x")

Baseline Model:               time = 37.87s throughput = 6.76 tok/s
Vector Index Embedding Model: time = 17.77s throughput = 14.41 tok/s
Speedup: 2.1314x


In [27]:
print("Text Baseline:")
print(text_ref)

print("\n\n")

print("Text Vector Index Embedding Model:")
print(text_vec)

Text Baseline:
The operation of a computer is a complex process involving a network of interconnected components, all working together to perform specific tasks. Here's a breakdown of the key components and how they work:

**1. Central Processing Unit (CPU):**

*   **Function:** The CPU is the "brain" of the computer. It executes instructions, which are the instructions that tell the computer what to do.
*   **How it works:** The CPU fetches instructions from memory, decodes them, and then executes them. It also manages the overall flow of data and logic.
*   **Key Components:**
    *   **Arithmetic Logic Unit (ALU):** Performs arithmetic and logical operations.
    *   **Control Unit:** Manages the flow of data and logic, making decisions.
    *   **Registers:** Small, fast storage locations within the CPU that hold the data and instructions that are currently being processed.

**2. Memory (RAM):**

*   **Function:** RAM is the computer's main storage area. It holds the data and instr

## Build a vector index embedding for a model

If we do not provide a prebuilt index for your desired model, you can easily generate one yourself. While the construction process takes some time, it only needs to be performed once, and the resulting index can be distributed alongside the model.
Below is our default index configuration when used with ef_construction = 5000. You can experiment with these parameters: higher values generally result in higher-quality indices but increase search latency.
Because approximate search might occasionally miss critical special tokens—such as the EOF token that terminates generation, it is recommended to add these tokens explicitly to the special_tokens list to have them always evaluated. The generated indices are saved to the data subfolder and can be loaded via: `VectorIndexEmbedding.from_file(f"data/<index_id>.index", ef=200)

In [28]:
model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"

# The amount of edges in the HNSW graph.
# 32 is a good tradeoff, higher leads to higher quality index, but slower search time.
M = 32

# Size of the candidate list during index construction.
# For production indices we use 5000 by default, which is very slow, but we only construct them once.
# For testing here, we just use 200 to get quick results.
ef_construction = 200

# Size of the candidate list evaluated during search.
ef = 200

# The number of nearest tokens used for top-k sampling.
k = 50

In [29]:
from vectorindex import VectorIndexEmbedding
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from vectorindex.vector_index import VectorIndexEmbeddingConfig

def get_special_tokens(model_id, tokenizer):
    add_tokenizer_special_tokens = True
    extra_tokens = []

    if "llama" in model_id:
        extra_tokens = ["'s", 'Ġand', 'Ġ**', ',', 'Ġto', ':**']
    if "Qwen3" in model_id:
        extra_tokens = ["\\n", "Ċ", "ĊĊ", ",", ".", "Ġ", "Ġor", "Ġto", "Ġand", "Ġa", "Ġof", "Ġ**"]
    if "google" in model_id:
        add_tokenizer_special_tokens = False
        extra_tokens = ["-", "▁of", "▁and", "▁the", "▁to", "▁a", ".", "▁▁", ",", "'", '"', ":**", "▁**", "\n", '▁"', "<pad>", "<eos>", "<bos>", "<unk>", "<mask>", "[multimodal]", "<start_of_turn>", "<end_of_turn>"]

    special_tokens = []
    if add_tokenizer_special_tokens:
        special_tokens.extend(list([k for k, v in tokenizer.added_tokens_decoder.items() if not "unused" in v.content and not "img_row" in v.content and not "reserved" in v.content and k < vocab_size]))
    special_tokens.extend([tokenizer.vocab[t] for t in extra_tokens])

    print(special_tokens)
    return special_tokens

set_seed(42)

model = AutoModelForCausalLM.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

weight = model.lm_head.weight.detach().clone().float()
vocab_size = weight.shape[0]

special_tokens = get_special_tokens(model_id, tokenizer)

index_id = VectorIndexEmbedding.get_index_name(model_id)
config = VectorIndexEmbeddingConfig(model_id=model_id, model_name=index_id, k=k, M=M, ef=ef, ef_construction=ef_construction, special_tokens=special_tokens)
print(f"Build index for config: {config}")
VectorIndexEmbedding.build_index(weight, config, save_path="data")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
Build index for config: VectorIndexEmbeddingConfig(model_name='huggingfacetb-smollm2-360m-instruct', k=50, ef=200, M=32, ef_construction=200, special_tokens=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16], dim=-1, vocab_size=-1, model_id='HuggingFaceTB/SmolLM2-360M-Instruct')
Index saved to data/huggingfacetb-smollm2-360m-instruct.index


'data/huggingfacetb-smollm2-360m-instruct.index'

In [30]:
from transformers import pipeline
from vectorindex import VectorIndexEmbedding

pipe = pipeline("text-generation", model=model_id, device="cpu")
pipe.model.lm_head = VectorIndexEmbedding.from_file(f"data/{VectorIndexEmbedding.get_index_name(model_id)}.index", ef=200)

prompt = [{"role": "user", "content": "What was at the beginning of time?"}]
out = pipe(prompt)
print(out[0]["generated_text"][-1]["content"])

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In the realm of Hatsune Mabuse's work, the concept of time is often somewhat of a mysterious and complex one.
